# 02. Integer optimization (Sec. VI-D)

The Sec. VI-D example of Censi (2015):

$$
x + y \geq \lceil\sqrt{x}\rceil + \lceil\sqrt{y}\rceil + c
$$

over $\mathbb{N} \times \mathbb{N}$. For each $c$ we want the Pareto-minimal set of $(x, y)$ pairs satisfying the inequality. The Kleene ascent starts from the seed $\{(0,0)\}$ and converges in a handful of steps. This is the example whose trace appears in Fig. 36 of the paper.


## Imports

In [1]:
import math
from codesign import (
    Antichain, FunctionDP, Loop, Ports, Naturals, solve,
)

## The model

The inner DP enumerates every splitting $(c_1, c_2)$ of the deficit into the two coordinates, giving the antichain of points $(x, y)$ with $x + y$ exactly meeting the constraint. The `Loop` on the axis `xy` closes $x_{out} \geq x_{in}$ and $y_{out} \geq y_{in}$ simultaneously.


In [2]:
def make_looped(c_value: int):
    N = Naturals()
    XY = Ports({"x": N, "y": N})
    F = Ports({"c": N, "xy": XY})
    R = Ports({"xy": XY, "xy_report": XY})

    def h(f):
        c = int(f["c"])
        x_in, y_in = f["xy"]["x"], f["xy"]["y"]
        if x_in == math.inf or y_in == math.inf:
            top = {"x": math.inf, "y": math.inf}
            return Antichain.singleton(R, {"xy": top, "xy_report": top})

        sx = math.isqrt(int(x_in)) + (1 if math.isqrt(int(x_in)) ** 2 < int(x_in) else 0)
        sy = math.isqrt(int(y_in)) + (1 if math.isqrt(int(y_in)) ** 2 < int(y_in) else 0)
        target = sx + sy + c

        pts = []
        for x_out in range(sx, target - sy + 1):
            y_out = target - x_out
            if y_out < sy:
                break
            pts.append({
                "xy": {"x": x_out, "y": y_out},
                "xy_report": {"x": x_out, "y": y_out},
            })
        if not pts:
            return Antichain.empty(R)
        return Antichain.from_set(R, pts)

    inner = FunctionDP(F=F, R=R, h_fn=h, name=f"sqrt_sum(c={c_value})")
    return Loop(inner, axis="xy")

## Run with trace

In [3]:
def pretty(p):
    return f"({p['xy_report']['x']}, {p['xy_report']['y']})"

def run(c_value, show_trace=True):
    looped = make_looped(c_value)
    result = solve(looped, {"c": c_value}, max_iter=50, record_trace=show_trace)
    print(f"c = {c_value}: iters = {result.iterations}, feasible = {result.feasible}")
    if show_trace and result.trace:
        for k, entry in enumerate(result.trace):
            A = entry.antichain
            pts = ", ".join(f"({p['xy']['x']}, {p['xy']['y']})" for p in A.points)
            print(f"   S_{k}: {{ {pts} }}")
    pts = ", ".join(pretty(p) for p in result.antichain.points)
    print(f"   M(c={c_value}) = {{ {pts} }}\n")
    return result

_ = run(0)
_ = run(1)
_ = run(4)
_ = run(20, show_trace=False)

c = 0: iters = 1, feasible = True
   S_0: { (0, 0) }
   S_1: { (0, 0) }
   M(c=0) = { (0, 0) }

c = 1: iters = 6, feasible = True
   S_0: { (0, 0) }
   S_1: { (0, 1), (1, 0) }
   S_2: { (0, 2), (1, 1), (2, 0) }
   S_3: { (0, 3), (1, 2), (2, 1), (3, 0) }
   S_4: { (0, 3), (2, 2), (3, 0) }
   S_5: { (0, 3), (3, 0) }
   S_6: { (0, 3), (3, 0) }
   M(c=1) = { (0, 3), (3, 0) }

c = 4: iters = 6, feasible = True
   S_0: { (0, 0) }
   S_1: { (0, 4), (1, 3), (2, 2), (3, 1), (4, 0) }
   S_2: { (0, 6), (1, 5), (2, 4), (4, 2), (5, 1), (6, 0) }
   S_3: { (0, 7), (1, 6), (3, 5), (4, 4), (5, 3), (6, 1), (7, 0) }
   S_4: { (0, 7), (2, 6), (4, 4), (6, 2), (7, 0) }
   S_5: { (0, 7), (3, 6), (4, 4), (6, 3), (7, 0) }
   S_6: { (0, 7), (3, 6), (4, 4), (6, 3), (7, 0) }
   M(c=4) = { (0, 7), (3, 6), (4, 4), (6, 3), (7, 0) }

c = 20: iters = 5, feasible = True
   M(c=20) = { (0, 25), (3, 24), (4, 23), (6, 22), (7, 21), (8, 20), (9, 19), (11, 18), (12, 16), (13, 15), (14, 14), (15, 13), (16, 12), (18, 11), (19

## Comments

The `c = 0` case is trivial: the empty mission has the solution $(0, 0)$.

For `c = 4` we get a five-point Pareto front $\{(0,7), (3,6), (4,4), (6,3), (7,0)\}$ in six iterations. Notice that the antichain is not necessarily monotone in cardinality: it grows, then shrinks, as some interior points get dominated by newly-discovered ones.

Note: the paper claims $M(1) = \{(1,0), (0,1)\}$, but those points fail the constraint ($1 \geq 1 + 0 + 1 = 2$ is false). The correct answer is $\{(0,3), (3,0)\}$, which is what our solver finds. The paper has a typo there.

The next notebook (**05**) renders these traces graphically.
